# Probabilidad de alcanzar Take Profit antes que Stop Loss en una estrategia sobre EUR/USD
### Comparación de regresión logística, máquina de vectores de soporte y perceptrón multicapa con optimización bayesiana

|                |   |
:----------------|---|
| **Fecha**      |   |
| **Materia**    | Laboratorio de Aprendizaje Estadístico |

| Alumno | Expediente |
|--------|------------|
| Juan Pablo Moran | 753760 |
|        |            |

## Objetivos

### General
Estimar la probabilidad de que una operación de una estrategia de trading sobre el par EUR/USD alcance su *take profit* (TP) antes que su *stop loss* (SL), a partir de variables de volatilidad, volumen y cambios de estructura del mercado, comparando tres modelos de clasificación con hiperparámetros optimizados para identificar cuál discrimina mejor entre operaciones ganadoras y perdedoras.

### Específicos
1. **Construir y analizar el conjunto de datos** a partir de precios históricos del EUR/USD: etiquetar cada señal de la estrategia según si tocó primero el TP o el SL y calcular los predictores (volatilidad, volumen y estructura) usando solo información disponible al momento de la entrada.
2. **Diseñar un pipeline de preprocesamiento** (limpieza, transformación de variables categóricas y escalamiento) aplicado de la misma forma a los tres modelos y ajustado únicamente con datos de entrenamiento en cada fold.
3. **Ajustar una regresión logística con regularización**, una **SVM con kernel RBF** y un **perceptrón multicapa**, optimizando los hiperparámetros de cada uno mediante optimización bayesiana con procesos gaussianos.
4. **Comparar los modelos** con el ROC-AUC (media y desviación estándar en validación cruzada k-folds con $k=10$), complementado con precisión, recall y F1.
5. **Evaluar la utilidad práctica del mejor modelo** comparando su probabilidad estimada contra la tasa de acierto mínima que necesita la estrategia para ser rentable dada su relación riesgo/beneficio.

## Marco teórico

### Regresión logística
La regresión logística modela la probabilidad de que una respuesta binaria $Y$ tome el valor 1 dados los predictores $X$. En lugar de ajustar una recta, pasa la combinación lineal de los predictores por la función sigmoide, que siempre queda entre 0 y 1:

$$ p(X) = P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \cdots + \beta_p X_p)}} $$

Despejando se obtiene que el logaritmo de los momios (*log-odds*) es lineal en los predictores:

$$ \ln\left(\frac{p(X)}{1-p(X)}\right) = \beta_0 + \beta_1 X_1 + \cdots + \beta_p X_p $$

Los coeficientes se estiman por máxima verosimilitud, lo que equivale a minimizar la entropía cruzada (*log-loss*):

$$ J(\vec{\beta}) = -\sum_{i=1}^n \left[ y_i \ln(\hat{p}(x_i)) + (1-y_i)\ln(1-\hat{p}(x_i)) \right] $$

Esta función es convexa, por lo que tiene un único mínimo global. La frontera de decisión ($p(X)=0.5$) es un hiperplano, es decir, el modelo solo separa clases de forma lineal en el espacio de los predictores.

**Regularización.** Igual que en Ridge y Lasso, se agrega una penalización a los coeficientes para reducir varianza y sobreajuste. En `sklearn` se controla con $C = 1/\lambda$ (un $C$ pequeño penaliza más) y con `l1_ratio`, que mezcla las penalizaciones L1 y L2 (*elastic net*):

$$ J_{reg}(\vec{\beta}) = J(\vec{\beta}) + \frac{1}{C}\left[ \alpha \sum_{j=1}^p |\beta_j| + \frac{1-\alpha}{2} \sum_{j=1}^p \beta_j^2 \right] $$

Con $\alpha = 1$ es Lasso (puede llevar coeficientes a cero y seleccionar variables) y con $\alpha = 0$ es Ridge (encoge todos los coeficientes sin eliminarlos).

### Máquinas de vectores de soporte (SVM)
Una SVM busca el hiperplano $f(x) = w^T x + b$ que separa las clases con el **margen** más grande posible, donde el margen es la distancia entre el hiperplano y las observaciones más cercanas de cada clase. Esas observaciones son los **vectores de soporte** y son las únicas que definen la frontera.

Como en datos reales las clases casi nunca son separables, se usa el clasificador de margen suave, que permite que algunas observaciones queden dentro del margen o del lado equivocado mediante variables de holgura $\xi_i$:

$$ \min_{w, b, \xi} \ \frac{1}{2} w^T w + C \sum_{i=1}^{n} \xi_i \quad \text{s.a.} \quad y_i(w^T \varphi(x_i) + b) \geq 1 - \xi_i, \ \ \xi_i \geq 0 $$

El hiperparámetro $C$ controla el balance: un $C$ grande castiga mucho los errores (margen angosto, menos sesgo y más varianza) y un $C$ pequeño permite más errores (margen amplio, más sesgo y menos varianza).

La SVM no da probabilidades directamente: su salida es una distancia al hiperplano. Esta distancia sirve para ordenar observaciones y calcular el ROC-AUC; si se necesitan probabilidades se calibran aparte (escalamiento de Platt).

### Kernels
Al resolver el problema dual de la SVM, los datos solo aparecen a través de productos punto $\varphi(x_i)^T\varphi(x_j)$. El **truco del kernel** reemplaza ese producto por una función $K(x_i, x_j)$ que equivale a mapear los datos a un espacio de mayor dimensión (un espacio de Hilbert) sin calcular el mapeo explícitamente. Así, una frontera lineal en el espacio transformado es no lineal en el espacio original:

$$ f(x) = \sum_{i \in S} \alpha_i y_i K(x_i, x) + b $$

Kernels comunes:

| Kernel | $K(x_i, x_j)$ |
|---|---|
| Lineal | $x_i^T x_j$ |
| Polinomial | $(\gamma\, x_i^T x_j + r)^d$ |
| RBF (gaussiano) | $\exp(-\gamma \lVert x_i - x_j \rVert^2)$ |
| Sigmoide | $\tanh(\gamma\, x_i^T x_j + r)$ |

En este proyecto se usa el kernel **RBF**, que mide similitud según la distancia: dos puntos cercanos tienen $K \approx 1$ y dos lejanos $K \approx 0$. El hiperparámetro $\gamma$ define qué tan local es esa similitud: un $\gamma$ grande hace fronteras muy flexibles (riesgo de sobreajuste) y uno pequeño fronteras suaves. Por eso $C$ y $\gamma$ se optimizan juntos. Como el kernel depende de distancias, es necesario escalar los predictores.

El kernel RBF también es el que usa el proceso gaussiano de la optimización bayesiana (ver más abajo).

### Redes neuronales: perceptrón multicapa (MLP)
Un perceptrón multicapa encadena varias capas de neuronas. Cada neurona calcula una combinación lineal de sus entradas y le aplica una función de activación no lineal:

$$ a^{(l)} = g\left(W^{(l)} a^{(l-1)} + b^{(l)}\right) $$

Con activación logística, una sola neurona es exactamente una regresión logística; el MLP es entonces una red de regresiones logísticas que se alimentan entre sí. Esto le permite resolver problemas no linealmente separables como el XOR. Activaciones comunes: logística, $\tanh$ y ReLU ($\max(0, z)$). La capa de salida usa una sigmoide para dar la probabilidad de la clase positiva.

Los pesos se ajustan minimizando la log-loss (más una penalización L2 controlada por `alpha`) con descenso de gradiente y retropropagación (*backpropagation*), usando optimizadores como `adam`, `sgd` o `lbfgs`. A diferencia de la regresión logística, al combinar varias capas la función de costo deja de ser convexa: tiene mínimos locales y el resultado depende de la inicialización (`random_state`), por lo que se fija la semilla para que los resultados sean reproducibles.

Hiperparámetros principales: número de capas ocultas y neuronas por capa (`hidden_layer_sizes`), función de activación, `alpha`, tasa de aprendizaje inicial y número máximo de iteraciones.

### Métricos para clasificación
Una clasificación binaria tiene cuatro resultados posibles, que se resumen en la matriz de confusión:

| | Predicho TP (1) | Predicho SL (0) |
|---|---|---|
| **Real TP (1)** | Verdadero positivo (VP) | Falso negativo (FN) |
| **Real SL (0)** | Falso positivo (FP) | Verdadero negativo (VN) |

- **Accuracy** $= \frac{VP + VN}{n}$: proporción de aciertos. Engaña cuando las clases están desbalanceadas.
- **Precisión** $= \frac{VP}{VP + FP}$: de las operaciones que el modelo dice que llegan a TP, cuántas realmente llegan. En trading es la tasa de acierto de las operaciones que sí se tomarían.
- **Recall (sensibilidad, TPR)** $= \frac{VP}{VP + FN}$: de las operaciones que llegaron a TP, cuántas detectó el modelo.
- **F1** $= 2 \cdot \frac{\text{precisión} \cdot \text{recall}}{\text{precisión} + \text{recall}}$: media armónica de las dos anteriores.
- **ROC-AUC**: la curva ROC grafica la tasa de verdaderos positivos contra la tasa de falsos positivos ($FPR = \frac{FP}{FP+VN}$) para todos los umbrales posibles. El área bajo la curva es la probabilidad de que el modelo le asigne mayor puntaje a una operación ganadora elegida al azar que a una perdedora. Vale 0.5 para un modelo que adivina y 1 para uno perfecto.

**Métrico elegido: ROC-AUC.** El objetivo es estimar una *probabilidad* y ordenar las operaciones de más a menos prometedoras, no fijar un umbral desde el inicio. El AUC no depende del umbral ni del balance de clases (que cambia según la relación TP/SL), y permite comparar los tres modelos en igualdad. Una vez elegido el modelo, el umbral se fija con la tasa de acierto mínima de la estrategia y se reporta la precisión en ese umbral.

### Hiperparámetros
Los **parámetros** son los valores que el modelo aprende de los datos al ajustarse (coeficientes $\beta$, pesos de la red, vectores de soporte). Los **hiperparámetros** se fijan antes del entrenamiento y controlan cómo aprende el modelo y qué tan complejo puede ser (el $C$ de la regularización, el $\gamma$ del kernel, el número de neuronas, `alpha`). No se pueden estimar con `fit`, así que se eligen probando combinaciones y evaluando cada una con validación cruzada.

| Modelo | Hiperparámetros optimizados | Rango de búsqueda |
|---|---|---|
| Regresión logística | $C$, `l1_ratio` | $C \in [10^{-4}, 10^{2}]$, `l1_ratio` $\in [0, 1]$ |
| SVM RBF | $C$, $\gamma$ | $C \in [10^{-2}, 10^{3}]$, $\gamma \in [10^{-4}, 10^{1}]$ |
| MLP | neuronas capa 1, neuronas capa 2, `alpha` | $[5, 50]$, $[5, 50]$, $[10^{-5}, 10^{0}]$ |

Métodos de búsqueda:
- **Grid search**: prueba todas las combinaciones de una malla. Es exhaustivo pero el costo crece exponencialmente con el número de hiperparámetros.
- **Random search**: prueba combinaciones al azar. Explora mejor espacios grandes, pero no aprende de los resultados anteriores.
- **Optimización bayesiana**: usa los resultados anteriores para decidir dónde probar después (siguiente sección).

### Optimización bayesiana
La optimización bayesiana sirve para optimizar funciones caras de evaluar y de las que no se conoce su forma. Aquí la función es

$$ f(\theta) = \text{AUC promedio en 10-fold CV del modelo con hiperparámetros } \theta $$

Cada evaluación implica entrenar el modelo 10 veces, así que se busca encontrar un buen $\theta$ con pocas evaluaciones. Se basa en el teorema de Bayes: se parte de una creencia previa sobre $f$ y se actualiza con cada nueva evaluación (evidencia) para obtener una creencia posterior.

**Proceso gaussiano (GP).** Es el modelo sustituto (*surrogate*) de $f$. Supone que los valores de $f$ en cualquier conjunto de puntos siguen una distribución normal multivariada, cuya covarianza la define un kernel; en este proyecto el RBF:

$$ k(\theta, \theta') = \sigma^2 \exp\left(-\frac{\lVert \theta - \theta' \rVert^2}{2\ell^2}\right) $$

Esto significa que hiperparámetros parecidos deben dar AUC parecidos. Con los puntos ya evaluados, el GP da para cada candidato una predicción $\mu(\theta)$ y una incertidumbre $\sigma(\theta)$: cerca de los puntos evaluados la incertidumbre es baja y lejos es alta.

**Función de adquisición.** Decide cuál es el siguiente punto a evaluar. Se usa el límite superior de confianza (*Upper Confidence Bound*), el mismo criterio visto en clase:

$$ \theta_{siguiente} = \arg\max_\theta \ \left[\mu(\theta) + 2\sigma(\theta)\right] $$

Este criterio balancea **explotación** (probar donde el AUC predicho es alto) y **exploración** (probar donde hay mucha incertidumbre).

**Algoritmo:**
1. Evaluar $f$ en unos pocos puntos al azar.
2. Ajustar el GP con los puntos evaluados.
3. Elegir el siguiente punto con la función de adquisición y evaluarlo.
4. Repetir 2 y 3 un número fijo de iteraciones y quedarse con el mejor $\theta$ encontrado.

Los hiperparámetros que varían en órdenes de magnitud ($C$, $\gamma$, `alpha`) se buscan en escala $\log_{10}$, para que el GP trate igual un cambio de 0.001 a 0.01 que uno de 10 a 100.

### Tema del proyecto: probabilidad de TP antes que SL en EUR/USD
**El mercado.** EUR/USD es el par de divisas más operado del mundo. Su precio se registra en velas (apertura, máximo, mínimo, cierre) a una temporalidad fija. Al ser un mercado descentralizado, el volumen disponible en los datos históricos suele ser **volumen de ticks** (número de cambios de precio por vela), que se usa como aproximación de la actividad del mercado.

**La operación.** Cada vez que se cumplen las reglas de la estrategia se abre una operación con dos niveles de salida:
- **Take profit (TP):** nivel donde se cierra con ganancia.
- **Stop loss (SL):** nivel donde se cierra con pérdida.

**Etiquetado (método de barreras).** Para cada señal se recorren las velas siguientes y se registra qué nivel se toca primero; si ninguno se toca en un número máximo de velas, la operación se descarta o se etiqueta por su resultado al cierre. Esto es una versión del método de triple barrera de López de Prado (2018). La variable respuesta queda:

$$ Y = \begin{cases} 1 & \text{si el precio toca el TP antes que el SL} \\ 0 & \text{si toca el SL antes que el TP} \end{cases} $$

**Predictores.** Se calculan al momento de la entrada usando solo velas pasadas, para no filtrar información del futuro (*look-ahead bias*):
- **Volatilidad:** *Average True Range* (ATR) y desviación estándar de los rendimientos en una ventana. Determina qué tan lejos queda el TP/SL en términos del movimiento normal del precio.
- **Volumen:** volumen de ticks relativo a su promedio móvil. Volumen alto suele acompañar movimientos con continuación.
- **Cambios de estructura:** rompimientos de máximos o mínimos recientes (*break of structure*), que indican si la tendencia sigue o se está revirtiendo.

**Relación con la rentabilidad.** Si el TP está a $R$ veces la distancia del SL, una operación gana $R$ unidades con probabilidad $p$ y pierde 1 con probabilidad $1-p$. El valor esperado es positivo cuando

$$ pR - (1-p) > 0 \quad \Longrightarrow \quad p > \frac{1}{1+R} $$

Por ejemplo, con $R = 2$ basta con acertar más del 33.3% de las veces. Un modelo útil es el que ordena bien las operaciones (AUC alto) y permite filtrar las que tienen probabilidad estimada por encima de ese umbral.

**Validación con series de tiempo.** Las observaciones están ordenadas en el tiempo y operaciones cercanas pueden traslaparse. Por eso los k-folds se hacen **sin revolver** los datos (`shuffle=False`): cada fold de prueba es un bloque de tiempo continuo, lo que reduce la fuga de información entre entrenamiento y prueba.

## Análisis del dataset

**¿De dónde viene?**
- *pendiente: fuente de los históricos de EUR/USD (broker/plataforma), temporalidad de las velas y periodo que cubren*

**¿Qué contiene?**
- *pendiente: columnas originales (fecha, apertura, máximo, mínimo, cierre, volumen) y número de velas*
- A partir de las velas se construye una tabla donde cada fila es una señal de la estrategia

**¿Qué información dan las muestras?**
- Cada muestra es una operación: el contexto del mercado al momento de entrar (volatilidad, volumen, estructura, dirección, hora) y si terminó en TP o en SL

**¿Qué se quiere analizar?**
- Si con la información disponible al entrar se puede estimar la probabilidad de que la operación llegue al TP antes que al SL, y qué modelo lo hace mejor

**¿Qué variables se tienen que transformar?**
- *pendiente de confirmar con las columnas finales*
- La fecha/hora no entra directo: se usa para ordenar y se extrae la sesión de mercado (Asia, Londres, Nueva York) como categórica
- Variables categóricas (dirección compra/venta, sesión, tipo de cambio de estructura) → variables dummy
- El target "TP"/"SL" → 1 y 0
- Las variables numéricas están en escalas muy distintas (el ATR es del orden de 0.001 y el volumen de miles) → estandarización con `StandardScaler`. Es necesario para la SVM (distancias) y el MLP (gradiente), y para que la regularización de la regresión logística trate igual a todos los coeficientes

**¿Qué resultado se podría encontrar?**
- Los mercados financieros son muy ruidosos, así que se espera un AUC moderado (cercano a 0.55–0.65). Un AUC de 0.5 indicaría que las variables no aportan información sobre el resultado de la operación
- Si la SVM o el MLP superan claramente a la regresión logística, la relación entre las variables y el resultado es no lineal

## Librerías

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

## Carga de datos

In [ ]:
df = pd.read_csv("eurusd_operaciones.csv")
print(df.shape)
df.head()

## Análisis exploratorio

In [ ]:
df.info()

In [ ]:
print(df.isnull().sum())
print(df.duplicated().sum())

In [ ]:
df.describe()

In [ ]:
numericas = ["atr", "volatilidad", "volumen_rel", "distancia_estructura"]
for col in numericas:
    plt.figure()
    plt.hist(df[col])
    plt.xlabel(col)
    plt.ylabel("Frecuencia")

In [ ]:
for col in numericas:
    plt.figure()
    plt.boxplot(df[col])
    plt.xlabel(col)

In [ ]:
categoricas = ["direccion", "sesion", "cambio_estructura", "resultado"]
for col in categoricas:
    print(col, df[col].unique())

In [ ]:
plt.figure()
df["resultado"].value_counts().plot(kind="bar")
plt.xlabel("resultado")
plt.ylabel("Frecuencia")

In [ ]:
df.groupby("resultado")[numericas].mean()

- *pendiente: observaciones del EDA*

## Limpieza y transformación de variables

In [ ]:
df = df.drop_duplicates()
df = df.dropna()

df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha")

df["resultado"] = df["resultado"].replace({"TP": 1, "SL": 0})
df["direccion"] = df["direccion"].replace({"compra": 1, "venta": 0})

for col in ["sesion", "cambio_estructura"]:
    valores = df[col].unique()
    for v in valores[1:]:
        df[col + "_" + v] = (df[col] == v).astype(int)

columnas = list(df.columns)
for col in ["fecha", "sesion", "cambio_estructura"]:
    columnas.remove(col)
df = df[columnas]

df = df.astype(float)
print(df.shape)
df.head()

- Los datos se ordenan por fecha para que los folds sean bloques de tiempo continuos
- `resultado` y `direccion` se pasan a 1 y 0
- `sesion` y `cambio_estructura` se pasan a variables dummy (una columna por categoría menos una)

In [ ]:
columnas = list(df.columns)
columnas.remove("resultado")
X = df[columnas]
y = df["resultado"]
print(X.shape)
print(y.mean())

## Pipeline

<center><img src="pipeline.png" width="900px"></center>

- El escalamiento va dentro del pipeline, así en cada fold el `StandardScaler` se ajusta solo con los datos de entrenamiento de ese fold y no hay fuga de información hacia el fold de prueba
- Los tres modelos usan exactamente el mismo preprocesamiento y los mismos 10 folds

In [ ]:
kf = KFold(n_splits=10)

## Modelos propuestos

En los tres casos la optimización bayesiana sigue el procedimiento visto en clase:
1. Se crea una malla de candidatos en el espacio de hiperparámetros
2. Se evalúan 5 puntos al azar con el AUC promedio de 10 folds
3. Se ajusta un proceso gaussiano con kernel RBF a los puntos evaluados
4. Se evalúa el candidato con mayor $\mu + 2\sigma$ y se repite 20 veces

### 1. Regresión logística con regularización
Hiperparámetros: $\log_{10}(C) \in [-4, 2]$ y `l1_ratio` $\in [0, 1]$ (0 = Ridge, 1 = Lasso).

In [ ]:
candidatos_lr = []
for logC in np.linspace(-4, 2, 25):
    for l1 in np.linspace(0, 1, 11):
        candidatos_lr.append([logC, l1])
candidatos_lr = np.array(candidatos_lr)
print(candidatos_lr.shape)

In [ ]:
np.random.seed(42)
idx = np.random.choice(len(candidatos_lr), 5, replace=False)
puntos_lr = candidatos_lr[idx]
auc_lr = []
for p in puntos_lr:
    modelo = make_pipeline(StandardScaler(),
                           LogisticRegression(C=10**p[0], l1_ratio=p[1], penalty="elasticnet",
                                              solver="saga", max_iter=5000))
    auc = cross_val_score(modelo, X, y, cv=kf, scoring="roc_auc")
    auc_lr.append(auc.mean())

for i in range(20):
    kernel = 1 * RBF(length_scale=1)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
    gp.fit(puntos_lr, auc_lr)
    y_pred, sigma = gp.predict(candidatos_lr, return_std=True)
    y_upper = y_pred + 2*sigma

    x_sig = candidatos_lr[np.argmax(y_upper)]
    modelo = make_pipeline(StandardScaler(),
                           LogisticRegression(C=10**x_sig[0], l1_ratio=x_sig[1], penalty="elasticnet",
                                              solver="saga", max_iter=5000))
    auc = cross_val_score(modelo, X, y, cv=kf, scoring="roc_auc")
    puntos_lr = np.vstack([puntos_lr, x_sig.reshape(1, -1)])
    auc_lr.append(auc.mean())

mejor_lr = puntos_lr[np.argmax(auc_lr)]
print("Mejor C:", 10**mejor_lr[0], " l1_ratio:", mejor_lr[1], " AUC:", max(auc_lr))

### 2. Máquina de vectores de soporte con kernel RBF
Hiperparámetros: $\log_{10}(C) \in [-2, 3]$ y $\log_{10}(\gamma) \in [-4, 1]$.

In [ ]:
candidatos_svm = []
for logC in np.linspace(-2, 3, 21):
    for logG in np.linspace(-4, 1, 21):
        candidatos_svm.append([logC, logG])
candidatos_svm = np.array(candidatos_svm)
print(candidatos_svm.shape)

In [ ]:
np.random.seed(42)
idx = np.random.choice(len(candidatos_svm), 5, replace=False)
puntos_svm = candidatos_svm[idx]
auc_svm = []
for p in puntos_svm:
    modelo = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10**p[0], gamma=10**p[1]))
    auc = cross_val_score(modelo, X, y, cv=kf, scoring="roc_auc")
    auc_svm.append(auc.mean())

for i in range(20):
    kernel = 1 * RBF(length_scale=1)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
    gp.fit(puntos_svm, auc_svm)
    y_pred, sigma = gp.predict(candidatos_svm, return_std=True)
    y_upper = y_pred + 2*sigma

    x_sig = candidatos_svm[np.argmax(y_upper)]
    modelo = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10**x_sig[0], gamma=10**x_sig[1]))
    auc = cross_val_score(modelo, X, y, cv=kf, scoring="roc_auc")
    puntos_svm = np.vstack([puntos_svm, x_sig.reshape(1, -1)])
    auc_svm.append(auc.mean())

mejor_svm = puntos_svm[np.argmax(auc_svm)]
print("Mejor C:", 10**mejor_svm[0], " gamma:", 10**mejor_svm[1], " AUC:", max(auc_svm))

### 3. Perceptrón multicapa
Dos capas ocultas. Hiperparámetros: neuronas en capa 1 $\in [5, 50]$, neuronas en capa 2 $\in [5, 50]$ y $\log_{10}(\alpha) \in [-5, 0]$.

In [ ]:
candidatos_mlp = []
for n1 in range(5, 51, 5):
    for n2 in range(5, 51, 5):
        for logA in np.linspace(-5, 0, 11):
            candidatos_mlp.append([n1, n2, logA])
candidatos_mlp = np.array(candidatos_mlp)
print(candidatos_mlp.shape)

In [ ]:
np.random.seed(42)
idx = np.random.choice(len(candidatos_mlp), 5, replace=False)
puntos_mlp = candidatos_mlp[idx]
auc_mlp = []
for p in puntos_mlp:
    modelo = make_pipeline(StandardScaler(),
                           MLPClassifier(hidden_layer_sizes=(int(p[0]), int(p[1])), alpha=10**p[2],
                                         max_iter=1000, early_stopping=True, random_state=42))
    auc = cross_val_score(modelo, X, y, cv=kf, scoring="roc_auc")
    auc_mlp.append(auc.mean())

for i in range(20):
    kernel = 1 * RBF(length_scale=[10, 10, 1])
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
    gp.fit(puntos_mlp, auc_mlp)
    y_pred, sigma = gp.predict(candidatos_mlp, return_std=True)
    y_upper = y_pred + 2*sigma

    x_sig = candidatos_mlp[np.argmax(y_upper)]
    modelo = make_pipeline(StandardScaler(),
                           MLPClassifier(hidden_layer_sizes=(int(x_sig[0]), int(x_sig[1])), alpha=10**x_sig[2],
                                         max_iter=1000, early_stopping=True, random_state=42))
    auc = cross_val_score(modelo, X, y, cv=kf, scoring="roc_auc")
    puntos_mlp = np.vstack([puntos_mlp, x_sig.reshape(1, -1)])
    auc_mlp.append(auc.mean())

mejor_mlp = puntos_mlp[np.argmax(auc_mlp)]
print("Mejor capas:", (int(mejor_mlp[0]), int(mejor_mlp[1])), " alpha:", 10**mejor_mlp[2], " AUC:", max(auc_mlp))

- En el MLP el kernel del GP tiene un `length_scale` por dimensión, porque las neuronas van de 5 a 50 y $\log_{10}(\alpha)$ de -5 a 0

### Convergencia de la optimización bayesiana

In [ ]:
plt.figure()
plt.plot(np.maximum.accumulate(auc_lr), label="Regresion logistica")
plt.plot(np.maximum.accumulate(auc_svm), label="SVM RBF")
plt.plot(np.maximum.accumulate(auc_mlp), label="MLP")
plt.axvline(4, color="gray", linestyle="--")
plt.xlabel("Evaluacion")
plt.ylabel("Mejor AUC encontrado")
plt.legend()
plt.grid()

- Las primeras 5 evaluaciones (antes de la línea punteada) son al azar; después las elige el proceso gaussiano

## Ajuste de modelos y comparación

Con los mejores hiperparámetros de cada modelo se repite la validación cruzada de 10 folds y se reporta la media y la desviación estándar del ROC-AUC, junto con precisión, recall, F1 y accuracy.

In [ ]:
modelo_lr = make_pipeline(StandardScaler(),
                          LogisticRegression(C=10**mejor_lr[0], l1_ratio=mejor_lr[1], penalty="elasticnet",
                                             solver="saga", max_iter=5000))
modelo_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10**mejor_svm[0], gamma=10**mejor_svm[1]))
modelo_mlp = make_pipeline(StandardScaler(),
                           MLPClassifier(hidden_layer_sizes=(int(mejor_mlp[0]), int(mejor_mlp[1])),
                                         alpha=10**mejor_mlp[2], max_iter=1000, early_stopping=True,
                                         random_state=42))

modelos = [modelo_lr, modelo_svm, modelo_mlp]
nombres = ["Regresion logistica", "SVM RBF", "MLP"]
metricos = ["roc_auc", "precision", "recall", "f1", "accuracy"]

resultados = pd.DataFrame(index=nombres)
auc_folds = []
for m in metricos:
    media = []
    std = []
    for modelo in modelos:
        score = cross_val_score(modelo, X, y, cv=kf, scoring=m)
        media.append(score.mean())
        std.append(np.std(score))
        if m == "roc_auc":
            auc_folds.append(score)
    resultados[m + " media"] = media
    resultados[m + " std"] = std
resultados

In [ ]:
plt.figure()
plt.boxplot(auc_folds)
plt.xticks([1, 2, 3], nombres)
plt.axhline(0.5, color="red", linestyle="--")
plt.ylabel("ROC-AUC por fold")
plt.grid()

- *pendiente: comparación de resultados*

### Utilidad para la estrategia

Con la relación riesgo/beneficio $R$ de la estrategia, la tasa de acierto mínima para no perder es $\frac{1}{1+R}$. Se compara la tasa de acierto de todas las señales contra la de las señales que el mejor modelo clasifica con mayor probabilidad (predicciones fuera de muestra de cada fold).

In [ ]:
from sklearn.model_selection import cross_val_predict

R = 2
umbral = 1 / (1 + R)
print("Tasa de acierto minima:", umbral)
print("Tasa de acierto sin filtro:", y.mean())

modelo_final = make_pipeline(StandardScaler(),
                             LogisticRegression(C=10**mejor_lr[0], l1_ratio=mejor_lr[1], penalty="elasticnet",
                                                solver="saga", max_iter=5000))
prob = cross_val_predict(modelo_final, X, y, cv=kf, method="predict_proba")[:, 1]

for q in [0, 0.25, 0.5, 0.75]:
    corte = np.quantile(prob, q)
    filtro = prob >= corte
    print("Corte", round(corte, 3), "| operaciones:", filtro.sum(), "| tasa de acierto:", round(y[filtro].mean(), 3))

- *pendiente: cambiar `R` por la relación TP/SL real y `modelo_final` por el mejor modelo si no es la regresión logística (para la SVM hay que usar `SVC(probability=True)`)*

## Conclusiones

- *pendiente: concluir respecto a cada objetivo específico con los resultados obtenidos*

## Referencias

- Bergstra, J., & Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research, 13*, 281–305.
- Bradley, A. P. (1997). The use of the area under the ROC curve in the evaluation of machine learning algorithms. *Pattern Recognition, 30*(7), 1145–1159. https://doi.org/10.1016/S0031-3203(96)00142-2
- Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*(3), 273–297. https://doi.org/10.1007/BF00994018
- James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An introduction to statistical learning: With applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0
- López de Prado, M. (2018). *Advances in financial machine learning*. Wiley.
- Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., ... Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
- Rasmussen, C. E., & Williams, C. K. I. (2006). *Gaussian processes for machine learning*. MIT Press.
- Snoek, J., Larochelle, H., & Adams, R. P. (2012). Practical Bayesian optimization of machine learning algorithms. *Advances in Neural Information Processing Systems, 25*, 2951–2959.
- Zou, H., & Hastie, T. (2005). Regularization and variable selection via the elastic net. *Journal of the Royal Statistical Society: Series B, 67*(2), 301–320. https://doi.org/10.1111/j.1467-9868.2005.00503.x